In [2]:
import numpy as np
import math
from decimal import Decimal, getcontext
from functools import lru_cache

getcontext().prec = 150

@lru_cache(None)
def factorial_dec(n: int) -> Decimal:
    if n < 0:
        raise ValueError("Negative factorial not defined")
    if n <= 1:
        return Decimal(1)
    return Decimal(n) * factorial_dec(n - 1)

@lru_cache(None)
def comb_dec(n: int, k: int) -> Decimal:
    if k < 0 or k > n:
        return Decimal(0)
    return factorial_dec(n) / (factorial_dec(k) * factorial_dec(n - k))

@lru_cache(None)
def D_coef(n: int, j: int) -> Decimal:
    if j == 0:
        return Decimal(1)
    
    result = Decimal(0)
    for k in range(0, n + 2*(j-1) + 1):
        for l in range(0, j):
            term = (Decimal((-1)**k) * 
                   D_coef(n, l) * 
                   comb_dec(n + 2*l, k - (j-1) + l) * 
                   Decimal((n + 2*(j-1) - 2*k) ** (n + 2*(j-1) + 2)) / 
                   (Decimal(2) ** (n + 2*(j-1) + 2) * 
                    factorial_dec(n + 2*(j-1) + 2)))
            result += term
    return result

@lru_cache(None)
def A_coef(k: int, n: int, m: int) -> Decimal:
    if k < 0 or k > 2*m:
        return Decimal(0)
    
    result = Decimal(0)
    sign = Decimal((-1)**(k - m))
    
    for l in range(0, m + 1):
        comb_arg = k - m + l
        if comb_arg >= 0:
            term = (sign * 
                   D_coef(n, l) * 
                   comb_dec(n + 2*l, comb_arg))
            result += term
    
    return result

@lru_cache(None)
def W_coef(m: int, k: int) -> float:
    result = Decimal(0)
    for n in range(0, m + 1):
        if m - n >= 0:
            a_val = A_coef(k, 2*n, m - n)
            denom = (Decimal(2) ** (2*n)) * factorial_dec(2*n + 1)
            result += a_val / denom
    return float(result)

def get_weights(m: int):
    weights = []
    for k in range(-m, m + 1):
        idx = m - k
        weights.append(W_coef(m, idx))
    return np.array(weights)

def integrate_diff_scheme(f, a, b, J, m):
    h = (b - a) / J
    weights = get_weights(m)
    offset = m
    
    y_values = []
    for j in range(-m, J + m):
        x = a + (j + 0.5) * h
        y_values.append(f(x))
    
    y = np.array(y_values)
    
    integral = 0.0
    for j in range(J):
        idx_start = j + offset
        y_slice = y[idx_start - m:idx_start + m + 1]
        integral += np.sum(weights * y_slice)
    
    return integral * h

def integrand(x):
    x = float(x)
    
    if abs(x) < 1e-6:
        return 0.5 - x/4 + x*x/24
    
    return math.sin(x/2) / (math.exp(x) - 1)

def compute_integral_machine_precision():
    a, b = -1.0, 1.0
    
    m = 6         
    J = 512       
    
    I = integrate_diff_scheme(integrand, a, b, J, m)
    
    I_half = integrate_diff_scheme(integrand, a, b, J//2, m)
    p = 2*m + 2
    error_est = abs(I - I_half) / (2**p - 1)
    
    return I, error_est

I, error = compute_integral_machine_precision()

print(f"{I:.16f}")

1.0132939418694187


In [5]:
def get_I_coefs(m: int):
    W = get_weights(m)
    I_coefs = np.zeros(2*m + 1)
    for l in range(2*m + 1):
        I_coefs[l] = np.sum(W[2*m - l:2*m + 1])
    return I_coefs


def direct_splicing_sum(a_func, j_m, m=7):

    I_coefs = get_I_coefs(m)

    S_direct = 0.0
    for j in range(1, j_m - m + 1):
        S_direct += a_func(j)
    
    S_transition = 0.0
    for j in range(j_m - m + 1, j_m + m + 1):
        # Индекс для I: j + m - j_m - 1
        idx = j + m - j_m - 1
        if 0 <= idx <= 2*m:
            S_transition += (1 - I_coefs[idx]) * a_func(j)
    
    return S_direct + S_transition

def euler_mascheroni(j_m, m=7):
    
    def a_j(j):
        return 1.0 / j
    
    S = direct_splicing_sum(a_j, j_m, m)
    
    gamma = S - math.log(j_m + 0.5)
    
    return gamma

def compute_gamma_with_precision(target_error=1e-13):

    m = 11
    
    gamma_exact = 0.57721566490153286060651209008240243104215933593992
    
    prev_gamma = None
    j_m = 50 
    
    while True:
        gamma = euler_mascheroni(j_m, m)
        
        if prev_gamma is not None:
            p = 2*m + 2
            error_est = abs(gamma - prev_gamma) / (2**p - 1)
            
            error_exact = abs(gamma - gamma_exact)
            
            print(f"{j_m:6d} | {gamma:20.14f} | {error_exact:12.2e} | "
                  f"оценка: {error_est:.2e}")
            
            if error_exact < target_error:
                print("-" * 60)
                print(f"Достигнута требуемая точность {target_error:.0e}")
                break
        else:
            print(f"{j_m:6d} | {gamma:20.14f} | {'—':>12} | {'—':>20}")
        
        prev_gamma = gamma
        j_m *= 2 
    
    return gamma, error_exact

gamma, error = compute_gamma_with_precision(1e-13)

print("\n" + "=" * 60)
print("РЕЗУЛЬТАТ С ТОЧНОСТЬЮ 13 ЗНАКОВ")
print("=" * 60)
print(f"γ_E = {gamma:.16f}")
print(f"Погрешность: {error:.2e}")

gamma_exact = 0.57721566490153286060651209008240243104215933593992
print(f"\nДля сравнения (точное значение):")
print(f"γ_E = {gamma_exact:.16f}")
print(f"Разница: {abs(gamma - gamma_exact):.2e}")

# Демонстрация эффективности метода
print("\n" + "=" * 60)
print("СРАВНЕНИЕ С ПРЯМЫМ СУММИРОВАНИЕМ")
print("=" * 60)

# Прямое суммирование для n = j_m
j_m_final = 1600  # Последнее использованное j_m
S_direct = 0
for j in range(1, j_m_final + 1):
    S_direct += 1.0/j
gamma_direct = S_direct - math.log(j_m_final)

print(f"Прямое суммирование до n={j_m_final}:")
print(f"γ_E = {gamma_direct:.16f}")
print(f"Погрешность: {abs(gamma_direct - gamma_exact):.2e}")
print(f"\nМетод прямой сшивки (m=7, j_m={j_m_final//2}):")
print(f"γ_E = {gamma:.16f}")
print(f"Погрешность: {error:.2e}")
print(f"Выигрыш в точности: ~{abs(gamma_direct - gamma_exact)/error:.0f} раз")

    50 |     0.57721564109791 |            — |                    —
   100 |     0.57721565194085 |     1.30e-08 | оценка: 6.46e-16
   200 |     0.57721565812117 |     6.78e-09 | оценка: 3.68e-16
   400 |     0.57721566143123 |     3.47e-09 | оценка: 1.97e-16
   800 |     0.57721566314567 |     1.76e-09 | оценка: 1.02e-16
  1600 |     0.57721566401833 |     8.83e-10 | оценка: 5.20e-17
  3200 |     0.57721566445859 |     4.43e-10 | оценка: 2.62e-17
  6400 |     0.57721566467970 |     2.22e-10 | оценка: 1.32e-17
 12800 |     0.57721566479051 |     1.11e-10 | оценка: 6.60e-18
 25600 |     0.57721566484604 |     5.55e-11 | оценка: 3.31e-18
 51200 |     0.57721566487380 |     2.77e-11 | оценка: 1.65e-18
102400 |     0.57721566488755 |     1.40e-11 | оценка: 8.20e-19
204800 |     0.57721566489481 |     6.72e-12 | оценка: 4.33e-19
409600 |     0.57721566489799 |     3.54e-12 | оценка: 1.90e-19
819200 |     0.57721566489922 |     2.32e-12 | оценка: 7.31e-20
1638400 |     0.57721566490022 |    

KeyboardInterrupt: 